In [13]:
"""
Downstream Evaluation: Promoter Detection

This notebook evaluates the learned DNA representations of the best-performing pretrained 
Hybrid RetNet model on a downstream binary classification task.
The original model was trained using masked language modelling (MLM), 
where the objective was to reconstruct masked 5-mer tokens from surrounding DNA context. 
In this stage, the pretrained encoder is reused as a feature extractor to investigate whether the learned representations 
contain biologically meaningful sequence information.
"""

'\nDownstream Evaluation: Promoter Detection\n\nThis notebook evaluates the learned DNA representations of the best-performing pretrained \nHybrid RetNet model on a downstream binary classification task.\nThe original model was trained using masked language modelling (MLM), \nwhere the objective was to reconstruct masked 5-mer tokens from surrounding DNA context. \nIn this stage, the pretrained encoder is reused as a feature extractor to investigate whether the learned representations \ncontain biologically meaningful sequence information.\n'

In [14]:
# ── Imports ───────────────────────────────────────────────────────────────────

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import random
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import itertools

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


from datasets import load_dataset, get_dataset_config_names
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


# ── Reproducibility ──────────────────────────────────────────────────────────

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seed set to {SEED}")


# ── Device ───────────────────────────────────────────────────────────────────

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

Seed set to 42
Device: cuda


In [15]:
# ── Build identical 5-mer vocabulary ─────────────────────────────────────────

bases = ["A", "C", "G", "T"]

kmers = [
    "".join(k)
    for k in itertools.product(bases, repeat=5)
]

kmer2idx = {
    kmer: idx
    for idx, kmer in enumerate(kmers)
}

kmer2idx["MASK"] = 1024
kmer2idx["PAD"]  = 1025
kmer2idx["UNK"]  = 1026


VOCAB = kmer2idx
VOCAB_SIZE = len(VOCAB)


# Reverse mapping if needed later
idx2kmer = {
    idx: kmer
    for kmer, idx in VOCAB.items()
}


print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Example 5-mers:")
print(list(kmer2idx.items())[:5])

print("\nSpecial tokens:")
print("MASK:", VOCAB["MASK"])
print("PAD :", VOCAB["PAD"])
print("UNK :", VOCAB["UNK"])

Vocabulary size: 1027
Example 5-mers:
[('AAAAA', 0), ('AAAAC', 1), ('AAAAG', 2), ('AAAAT', 3), ('AAACA', 4)]

Special tokens:
MASK: 1024
PAD : 1025
UNK : 1026


In [16]:
# ── Model Hyperparameters (same as pretraining) ──────────────────────────────

WINDOW_SIZE = 128

EMBED_DIM   = 128
NUM_HEADS   = 4
FF_DIM      = 256
DROPOUT     = 0.1

GAMMAS = [
    1 - 2**(-5 - i)
    for i in range(NUM_HEADS)
]


# ── Multi Scale Retention ────────────────────────────────────────────────────

class MultiScaleRetention(nn.Module):

    def __init__(self, embed_dim, num_heads, gammas, dropout=DROPOUT):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.gammas = gammas

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "D",
            self._build_decay_matrix()
        )


    def _build_decay_matrix(self):

        n = WINDOW_SIZE

        D = torch.zeros(
            self.num_heads,
            n,
            n
        )

        for h, gamma in enumerate(self.gammas):
            for i in range(n):
                for j in range(i + 1):
                    D[h, i, j] = gamma ** (i-j)

        return D


    def forward(self, x):

        batch, seq_len, _ = x.shape

        Q = self.q_proj(x).view(
            batch, seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        K = self.k_proj(x).view(
            batch, seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        V = self.v_proj(x).view(
            batch, seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)


        D = self.D[:, :seq_len, :seq_len]

        QK = torch.matmul(
            Q,
            K.transpose(-2,-1)
        )

        QK = QK * D.unsqueeze(0)

        QK = QK / (seq_len ** 0.5)

        out = torch.matmul(QK,V)

        out = (
            out.transpose(1,2)
            .contiguous()
            .view(batch, seq_len, self.embed_dim)
        )

        return self.dropout(
            self.o_proj(out)
        )


# ── RetNet Block ─────────────────────────────────────────────────────────────

class RetNetBlock(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        gammas,
        ff_dim=FF_DIM,
        dropout=DROPOUT
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.retention = MultiScaleRetention(
            embed_dim,
            num_heads,
            gammas,
            dropout
        )

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )


    def forward(self,x):

        x = x + self.retention(
            self.norm1(x)
        )

        x = x + self.ff(
            self.norm2(x)
        )

        return x



# ── Attention Block ──────────────────────────────────────────────────────────

class AttentionBlock(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads=NUM_HEADS,
        ff_dim=FF_DIM,
        dropout=DROPOUT
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )


    def forward(self,x):

        normed = self.norm1(x)

        attn_out,_ = self.attn(
            normed,
            normed,
            normed
        )

        x = x + attn_out

        x = x + self.ff(
            self.norm2(x)
        )

        return x



# ── Original MLM Model ───────────────────────────────────────────────────────

class HybridRetNetModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            EMBED_DIM,
            padding_idx=VOCAB["PAD"]
        )

        self.pos_embedding = nn.Embedding(
            WINDOW_SIZE,
            EMBED_DIM
        )


        self.layers = nn.ModuleList([
            RetNetBlock(
                EMBED_DIM,
                NUM_HEADS,
                GAMMAS
            ),
            RetNetBlock(
                EMBED_DIM,
                NUM_HEADS,
                GAMMAS
            ),
            RetNetBlock(
                EMBED_DIM,
                NUM_HEADS,
                GAMMAS
            ),
            AttentionBlock(
                EMBED_DIM
            )
        ])


        self.norm = nn.LayerNorm(
            EMBED_DIM
        )


        self.output_layer = nn.Linear(
            EMBED_DIM,
            VOCAB_SIZE
        )


    def forward(self,x):

        positions = torch.arange(
            0,
            x.size(1),
            device=x.device
        ).unsqueeze(0)


        x = (
            self.embedding(x)
            +
            self.pos_embedding(positions)
        )


        for layer in self.layers:
            x = layer(x)


        x = self.norm(x)

        return self.output_layer(x)

In [17]:
# ── Load pretrained model checkpoint ─────────────────────────────────────────

model = HybridRetNetModel().to(DEVICE)

checkpoint_path = "hybrid_retnet_best_mr.pth"

state_dict = torch.load(
    checkpoint_path,
    map_location=DEVICE
)

model.load_state_dict(state_dict)

model.eval()

print("Pretrained Hybrid RetNet checkpoint loaded successfully.")

print(
    f"Total parameters: {sum(p.numel() for p in model.parameters()):,}"
)

Pretrained Hybrid RetNet checkpoint loaded successfully.
Total parameters: 808,963


In [18]:
# ── Encoder wrapper for downstream tasks ─────────────────────────────────────

class HybridRetNetEncoder(nn.Module):

    def __init__(self, pretrained_model):
        super().__init__()

        self.embedding = pretrained_model.embedding
        self.pos_embedding = pretrained_model.pos_embedding

        self.layers = pretrained_model.layers
        self.norm = pretrained_model.norm


    def forward(self, x):

        positions = torch.arange(
            0,
            x.size(1),
            device=x.device
        ).unsqueeze(0)


        # Token + positional embeddings
        x = (
            self.embedding(x)
            +
            self.pos_embedding(positions)
        )


        # Encoder layers
        for layer in self.layers:
            x = layer(x)


        x = self.norm(x)


        # Mean pooling:
        # (batch, sequence_length, embedding_dim)
        # ->
        # (batch, embedding_dim)

        sequence_embedding = x.mean(dim=1)


        return sequence_embedding



# Create encoder from pretrained MLM model

encoder = HybridRetNetEncoder(model).to(DEVICE)

encoder.eval()

print("Hybrid RetNet encoder created.")

Hybrid RetNet encoder created.


In [19]:
# Test encoder output size

dummy_input = torch.randint(
    0,
    VOCAB_SIZE,
    (2, WINDOW_SIZE)
).to(DEVICE)


with torch.no_grad():
    output = encoder(dummy_input)


print("Input shape :", dummy_input.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([2, 128])
Output shape: torch.Size([2, 128])


In [20]:

promoter_dataset = load_dataset(
    "leannmlindsey/GUE",
    name="prom_core_all"
)

print(promoter_dataset)

DatasetDict({
    train: Dataset({
        features: ['sequence', 'label'],
        num_rows: 47356
    })
    test: Dataset({
        features: ['sequence', 'label'],
        num_rows: 5920
    })
    dev: Dataset({
        features: ['sequence', 'label'],
        num_rows: 5920
    })
})


In [21]:
# Convert training split to pandas for inspection

train_df = promoter_dataset["train"].to_pandas()


print("First examples:")
display(train_df.head())


print("\nLabel distribution:")
print(train_df["label"].value_counts())


# Sequence lengths

lengths = train_df["sequence"].apply(len)


print("\nSequence length statistics:")
print(lengths.describe())


print("\nUnique sequence lengths:")
print(lengths.unique()[:20])

First examples:


,sequence,label
0,GCTAGCTCATCTTGCGGCTGGGCGGGGCCCAGGACTGCTGCTGCTG...,1
1,ATAAAGGGGGCATCTCGCAGCACCGGGGGCCCTAAGCAGCGAGACC...,1
2,GGGGAGCTCTGGGAACGGGGCCTGTGCGCACGCGCATCTGACGGTT...,1
3,GCCACCTGAGCGTAGGGCATACAGCCATTTTCTGGGCGGGGCGTGC...,0
4,GGAAAGAGCAGACAAACAGGAACAGAAGTTCTCACTCTAGGTCACG...,1



Label distribution:
label
0    23705
1    23651
Name: count, dtype: int64

Sequence length statistics:
count    47356.0
mean        70.0
std          0.0
min         70.0
25%         70.0
50%         70.0
75%         70.0
max         70.0
Name: sequence, dtype: float64

Unique sequence lengths:
[70]


In [23]:
# ── Core promoter PyTorch Dataset ────────────────────────────────────────────

class PromoterDataset(Dataset):

    def __init__(self, hf_dataset):

        self.sequences = hf_dataset["sequence"]
        self.labels = hf_dataset["label"]


    def tokenize_5mer(self, sequence):

        tokens = []

        sequence = sequence.upper()

        for i in range(len(sequence) - 4):

            kmer = sequence[i:i+5]

            if kmer in VOCAB:
                tokens.append(
                    VOCAB[kmer]
                )
            else:
                tokens.append(
                    VOCAB["UNK"]
                )

        return tokens


    def __len__(self):

        return len(self.sequences)


    def __getitem__(self, idx):

        tokens = self.tokenize_5mer(
            self.sequences[idx]
        )

        label = self.labels[idx]


        return (
            torch.tensor(
                tokens,
                dtype=torch.long
            ),
            torch.tensor(
                label,
                dtype=torch.float
            )
        )


# Create datasets

train_promoter = PromoterDataset(
    promoter_dataset["train"]
)

dev_promoter = PromoterDataset(
    promoter_dataset["dev"]
)

test_promoter = PromoterDataset(
    promoter_dataset["test"]
)


print("Train samples:", len(train_promoter))
print("Dev samples:", len(dev_promoter))
print("Test samples:", len(test_promoter))

Train samples: 47356
Dev samples: 5920
Test samples: 5920


In [24]:
# ── DataLoaders ──────────────────────────────────────────────────────────────

BATCH_SIZE = 64


train_loader_promoter = DataLoader(
    train_promoter,
    batch_size=BATCH_SIZE,
    shuffle=True
)

dev_loader_promoter = DataLoader(
    dev_promoter,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_promoter = DataLoader(
    test_promoter,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("Train batches:", len(train_loader_promoter))
print("Dev batches:", len(dev_loader_promoter))
print("Test batches:", len(test_loader_promoter))

Train batches: 740
Dev batches: 93
Test batches: 93


In [25]:
# ── Promoter classifier ──────────────────────────────────────────────────────

class PromoterClassifier(nn.Module):

    def __init__(self, encoder):

        super().__init__()

        self.encoder = encoder

        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )


    def forward(self, x):

        features = self.encoder(x)

        logits = self.classifier(features)

        return logits.squeeze(1)



# Create downstream model

promoter_model = PromoterClassifier(
    encoder
).to(DEVICE)


print(promoter_model)

print(
    f"\nTotal parameters: "
    f"{sum(p.numel() for p in promoter_model.parameters()):,}"
)

PromoterClassifier(
  (encoder): HybridRetNetEncoder(
    (embedding): Embedding(1027, 128, padding_idx=1025)
    (pos_embedding): Embedding(128, 128)
    (layers): ModuleList(
      (0-2): 3 x RetNetBlock(
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (retention): MultiScaleRetention(
          (q_proj): Linear(in_features=128, out_features=128, bias=False)
          (k_proj): Linear(in_features=128, out_features=128, bias=False)
          (v_proj): Linear(in_features=128, out_features=128, bias=False)
          (o_proj): Linear(in_features=128, out_features=128, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): Sequential(
          (0): Linear(in_features=128, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=256, out_features=128, bias=True)
     

In [26]:
# ── Training setup ───────────────────────────────────────────────────────────

from sklearn.metrics import accuracy_score, f1_score

EPOCHS = 10
LEARNING_RATE = 1e-4
PATIENCE = 2

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    promoter_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

best_loss = float("inf")
patience_counter = 0

In [27]:
# ── Train / Validation functions ─────────────────────────────────────────────

def train_epoch(model, loader):

    model.train()

    total_loss = 0

    preds = []
    labels = []

    for x, y in loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        probabilities = torch.sigmoid(logits)

        predictions = (probabilities >= 0.5).float()

        preds.extend(predictions.cpu().numpy())

        labels.extend(y.cpu().numpy())

    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return total_loss / len(loader), accuracy, f1


def evaluate(model, loader):

    model.eval()

    total_loss = 0

    preds = []
    labels = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item()

            probabilities = torch.sigmoid(logits)

            predictions = (probabilities >= 0.5).float()

            preds.extend(predictions.cpu().numpy())

            labels.extend(y.cpu().numpy())

    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return total_loss / len(loader), accuracy, f1

In [28]:
# ── Fine-tuning loop ─────────────────────────────────────────────────────────

for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc, train_f1 = train_epoch(
        promoter_model,
        train_loader_promoter
    )

    val_loss, val_acc, val_f1 = evaluate(
        promoter_model,
        dev_loader_promoter
    )

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_loss < best_loss:

        best_loss = val_loss
        patience_counter = 0

        torch.save(
            promoter_model.state_dict(),
            "best_promoter_model.pth"
        )

        print("✓ Best model saved.")

    else:

        patience_counter += 1

        print(
            f"No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print("\nEarly stopping triggered.")
            break

Epoch 01/10 | Train Loss: 0.4597 | Train Acc: 0.7819 | Train F1: 0.7763 | Val Loss: 0.4125 | Val Acc: 0.8166 | Val F1: 0.8093
✓ Best model saved.
Epoch 02/10 | Train Loss: 0.3718 | Train Acc: 0.8367 | Train F1: 0.8364 | Val Loss: 0.3727 | Val Acc: 0.8353 | Val F1: 0.8373
✓ Best model saved.
Epoch 03/10 | Train Loss: 0.3397 | Train Acc: 0.8531 | Train F1: 0.8533 | Val Loss: 0.3645 | Val Acc: 0.8458 | Val F1: 0.8485
✓ Best model saved.
Epoch 04/10 | Train Loss: 0.3178 | Train Acc: 0.8637 | Train F1: 0.8643 | Val Loss: 0.3611 | Val Acc: 0.8454 | Val F1: 0.8419
✓ Best model saved.
Epoch 05/10 | Train Loss: 0.3005 | Train Acc: 0.8731 | Train F1: 0.8739 | Val Loss: 0.3508 | Val Acc: 0.8525 | Val F1: 0.8544
✓ Best model saved.
Epoch 06/10 | Train Loss: 0.2849 | Train Acc: 0.8806 | Train F1: 0.8810 | Val Loss: 0.3502 | Val Acc: 0.8527 | Val F1: 0.8533
✓ Best model saved.
Epoch 07/10 | Train Loss: 0.2708 | Train Acc: 0.8866 | Train F1: 0.8873 | Val Loss: 0.3566 | Val Acc: 0.8486 | Val F1: 0.845

In [29]:
# ── Load best checkpoint ─────────────────────────────────────────────────────

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import numpy as np


# Load best validation checkpoint

promoter_model.load_state_dict(
    torch.load(
        "best_promoter_model.pth",
        map_location=DEVICE
    )
)

promoter_model.eval()


all_predictions = []
all_probabilities = []
all_labels = []


with torch.no_grad():

    for x, y in test_loader_promoter:

        x = x.to(DEVICE)

        logits = promoter_model(x)

        probabilities = torch.sigmoid(logits)

        predictions = (
            probabilities >= 0.5
        ).float()


        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

        all_labels.extend(
            y.numpy()
        )


# Convert arrays

all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)
all_labels = np.array(all_labels)


# Metrics

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

test_f1 = f1_score(
    all_labels,
    all_predictions
)

test_mcc = matthews_corrcoef(
    all_labels,
    all_predictions
)

test_auc = roc_auc_score(
    all_labels,
    all_probabilities
)


print("=" * 60)
print(" GUE Core Promoter Detection - Test Results")
print("=" * 60)

print(f"Accuracy : {test_accuracy:.4f}")
print(f"F1-score : {test_f1:.4f}")
print(f"MCC      : {test_mcc:.4f}")
print(f"AUROC    : {test_auc:.4f}")

print("=" * 60)

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        digits=4
    )
)

 GUE Core Promoter Detection - Test Results
Accuracy : 0.8514
F1-score : 0.8515
MCC      : 0.7027
AUROC    : 0.9294

Classification Report:
              precision    recall  f1-score   support

         0.0     0.8495    0.8530    0.8513      2952
         1.0     0.8532    0.8497    0.8515      2968

    accuracy                         0.8514      5920
   macro avg     0.8514    0.8514    0.8514      5920
weighted avg     0.8514    0.8514    0.8514      5920

